# 03 — Radiomics + VaRFS

**Inputs needed:** `data/processed/<PID>/before_cropped.nii.gz` and `before_liver.nii.gz` per patient, plus `data/labels.csv`.
**Outputs produced:** `data/raw_radiomics_features.csv`, `data/varfs_filtered_features.csv`, `data/varfs_selected_features.json`, `models/saved/radiomics_baseline.pkl`, `results/baseline_metrics.json`.
**Runtime:** ~30 s per patient for PyRadiomics; ~30 s for the bootstrap stability filter.


Phase 3 walkthrough:

1. Run `RadiomicsExtractor` on every patient's cropped liver volume → `data/raw_radiomics_features.csv`.
2. Apply `VaRFSSelector` (bootstrap stability + correlation pruning) → `data/varfs_filtered_features.csv` and `data/varfs_selected_features.json`.
3. Train the classical `RadiomicsBaseline` (RandomForest) and inspect the cross-validated metrics.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import ensure_dirs, load_config, load_features_config, set_seed
from src.utils.logger import setup_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
features_cfg = load_features_config(ROOT / "configs" / "features.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "radiomics.log")

In [ ]:
from src.features.radiomics_extractor import RadiomicsExtractor

raw_csv = ROOT / "data" / "raw_radiomics_features.csv"
extractor = RadiomicsExtractor(features_cfg["radiomics"])
raw_features = extractor.extract_all(
    processed_dir=cfg["paths"]["processed_dir"],
    labels_csv=cfg["paths"]["labels_csv"],
    output_csv=raw_csv,
)
raw_features.shape

In [ ]:
from src.features.varfs_selection import VaRFSSelector

varfs_cfg = features_cfg["varfs"]
selector = VaRFSSelector(
    n_bootstrap=int(varfs_cfg["n_bootstrap"]),
    stability_threshold=float(varfs_cfg["stability_threshold"]),
    correlation_threshold=float(varfs_cfg["correlation_threshold"]),
    top_k_per_iter=int(varfs_cfg["top_k_per_iter"]),
    use_robust_scaler=bool(varfs_cfg["use_robust_scaler"]),
    random_state=int(cfg["seed"]),
)
selector.fit(raw_features, raw_features["label"])
filtered = selector.transform(raw_features)
filtered.to_csv(ROOT / "data" / "varfs_filtered_features.csv", index=False)
selector.save_selection(ROOT / "data" / "varfs_selected_features.json")
print(f"Stable features: {len(selector.selected_features_)} / {raw_features.shape[1] - 2}")

In [ ]:
import pandas as pd
stability = pd.Series(selector.stability_scores_).sort_values(ascending=False)
stability.head(20).plot.barh(figsize=(7, 6), title="Top 20 stability scores")

In [ ]:
from src.features.baseline_classifier import RadiomicsBaseline, save_metrics

baseline = RadiomicsBaseline(features_cfg["baseline"])
metrics = baseline.train(filtered)
baseline.save(Path(cfg["paths"]["model_save_dir"]) / "radiomics_baseline.pkl")
save_metrics(metrics, Path(cfg["paths"]["results_dir"]) / "baseline_metrics.json")
metrics